## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
#include <bits/stdc++.h>
using namespace std;

class FastInput {
public:
    int nextInt() {
        int x = 0;
        int c = getchar();

        while (c != EOF && (c < '0' || c > '9')) {
            c = getchar();
        }

        if (c == EOF) return 0;

        while (c >= '0' && c <= '9') {
            x = x * 10 + (c - '0');
            c = getchar();
        }

        return x;
    }
};

static const int LIM = 1050;

int n;
int fixedA, fixedB;
int blockSize;

vector<int> curPerm;
vector<int> whereIs;
vector<int> operations;

void refreshPosition() {
    for (int i = 0; i < n; ++i) {
        whereIs[curPerm[i]] = i;
    }
}

void recordFixedSwap() {
    operations.push_back(0);

    for (int &v : curPerm) {
        if (v == fixedA) v = fixedB;
        else if (v == fixedB) v = fixedA;
    }

    refreshPosition();
}

void recordAdd(int delta) {
    delta %= n;
    if (delta < 0) delta += n;
    if (delta == 0) return;

    operations.push_back(delta);

    for (int &v : curPerm) {
        v = (v + delta) % n;
    }

    refreshPosition();
}

void recordXor(int mask) {
    if (mask == 0) return;

    operations.push_back(-mask);

    for (int &v : curPerm) {
        v ^= mask;
    }

    refreshPosition();
}

pair<int, int> calcMappedPair(int x, int y) {
    int diff = (y - x + n - blockSize + n) % n;

    int px = 0;
    int py = 0;

    for (int step = n / 2; step >= 2 * blockSize; step >>= 1) {
        if (diff >= step) {
            diff -= step;
            py += step / 2;
        } else {
            px += step / 2;
        }
    }

    int low = x & (blockSize - 1);

    px += n / 2;
    px += low;
    py += low;

    return {px, py};
}

void makeSwapPossible(int x, int y);

void exchangeByMagic(int x, int y) {
    int bx = x / blockSize;
    int by = y / blockSize;

    if ((bx & 1) == (by & 1)) {
        int bridge;

        if ((bx & 1) == 0) {
            bridge = (x & (blockSize - 1)) + blockSize;
        } else {
            bridge = (x & (blockSize - 1));
        }

        makeSwapPossible(x, bridge);
        makeSwapPossible(y, bridge);
        makeSwapPossible(x, bridge);
        return;
    }

    auto [pa, pb] = calcMappedPair(fixedA, fixedB);
    auto [px, py] = calcMappedPair(x, y);

    recordAdd((px - x + n) % n);
    recordXor(px ^ pa);
    recordAdd((fixedA - pa + n) % n);

    recordFixedSwap();

    recordAdd((pa - fixedA + n) % n);
    recordXor(px ^ pa);
    recordAdd((x - px + n) % n);
}

void makeSwapPossible(int x, int y) {
    exchangeByMagic(x, y);
}

struct ReducedPermutation {
    int len;
    vector<int> value;
    vector<int> ops;

    explicit ReducedPermutation(int m = 0) {
        len = m;
        value.assign(m, 0);
    }

    bool build() {
        vector<int> seen(len, 0);

        for (int x : value) {
            if (x < 0 || x >= len) return false;
            seen[x] = 1;
        }

        for (int i = 0; i < len; ++i) {
            if (!seen[i]) return false;
        }

        if (len == 1) return true;

        int half = len / 2;

        ReducedPermutation evenPart(half);
        ReducedPermutation oddPart(half);

        for (int i = 0; i < half; ++i) {
            evenPart.value[i] = value[i * 2] / 2;
            oddPart.value[i] = value[i * 2 + 1] / 2;
        }

        if (!evenPart.build()) return false;
        if (!oddPart.build()) return false;

        if (value[0] & 1) {
            ops.push_back(len == 2 ? 1 : -1);
        }

        int leftXor = 0;

        for (int op : evenPart.ops) {
            if (op > 0) {
                ops.push_back(-1);
                ops.push_back(1);
            } else {
                int realMask = (-op) * 2;
                ops.push_back(-realMask);
                leftXor ^= realMask;
            }
        }

        if (leftXor) {
            ops.push_back(-leftXor);
        }

        int rightXor = 0;

        for (int op : oddPart.ops) {
            if (op > 0) {
                ops.push_back(1);
                ops.push_back(-1);
            } else {
                int realMask = (-op) * 2;
                ops.push_back(-realMask);
                rightXor ^= realMask;
            }
        }

        if ((leftXor & half) != (rightXor & half)) {
            return false;
        }

        if (leftXor >= half) leftXor -= half;
        if (rightXor >= half) rightXor -= half;

        if (leftXor != rightXor) {
            return false;
        }

        compactOps();
        return true;
    }

    void compactOps() {
        vector<int> merged;

        for (int op : ops) {
            if (!merged.empty() && op < 0 && merged.back() < 0) {
                int a = -merged.back();
                int b = -op;
                int nxt = a ^ b;

                if (nxt == 0) {
                    merged.pop_back();
                } else {
                    merged.back() = -nxt;
                }
            } else {
                merged.push_back(op);
            }
        }

        ops.swap(merged);
    }
};

bool checkResidueClass(int start) {
    vector<int> bucket;

    for (int i = start; i < n; i += blockSize) {
        bucket.push_back(curPerm[i]);
    }

    sort(bucket.begin(), bucket.end());

    int ptr = 0;

    for (int i = start; i < n; i += blockSize) {
        if (bucket[ptr++] != i) {
            return false;
        }
    }

    return true;
}

int main() {
    FastInput input;

    n = input.nextInt();
    if (n == 0) return 0;

    fixedA = input.nextInt();
    fixedB = input.nextInt();

    curPerm.assign(n, 0);
    whereIs.assign(n, 0);

    for (int i = 0; i < n; ++i) {
        curPerm[i] = input.nextInt();
    }

    refreshPosition();

    blockSize = (fixedA - fixedB + n) % n;
    blockSize &= -blockSize;

    if (blockSize == 0) {
        blockSize = n;
    }

    if (blockSize > 1) {
        ReducedPermutation base(blockSize);

        for (int i = 0; i < blockSize; ++i) {
            base.value[i] = curPerm[i] & (blockSize - 1);
        }

        if (!base.build()) {
            cout << -1 << '\n';
            return 0;
        }

        for (int op : base.ops) {
            if (op > 0) {
                recordAdd(op);
            } else {
                recordXor(-op);
            }
        }
    }

    for (int r = 0; r < blockSize; ++r) {
        if (!checkResidueClass(r)) {
            cout << -1 << '\n';
            return 0;
        }

        for (int i = r; i < n; i += blockSize) {
            if (curPerm[i] != i) {
                makeSwapPossible(i, curPerm[i]);
            }
        }
    }

    for (int i = 0; i < n; ++i) {
        assert(curPerm[i] == i);
    }

    cout << operations.size() << '\n';

    for (int op : operations) {
        if (op == 0) {
            cout << 0 << '\n';
        } else if (op < 0) {
            cout << "1 " << -op << '\n';
        } else {
            cout << "2 " << op << '\n';
        }
    }

    return 0;
}

## B 长跑

In [ ]:
#import sys
from collections import deque

def main():
    data = sys.stdin.read().split()
    if not data:
        return

    pointer = 0
    total_tokens = len(data)

    while pointer < total_tokens:
        if pointer + 3 >= total_tokens:
            break  # 数据不完整时直接退出

        num_stations = int(data[pointer])
        distance = int(data[pointer + 1])
        stamina_max = int(data[pointer + 2])
        budget = int(data[pointer + 3])
        pointer += 4

        # 读取补给站信息
        supply_points = {}
        for _ in range(num_stations):
            location = int(data[pointer])
            cost = int(data[pointer + 1])
            pointer += 2

            if location >= distance:
                continue

            # 同位置保留最小花费
            if location not in supply_points or cost < supply_points[location]:
                supply_points[location] = cost

        # 特殊情况：初始体力即可到终点
        if distance <= stamina_max:
            print("Yes")
            continue

        sorted_locations = sorted(supply_points.keys())
        num_valid_stations = len(sorted_locations)

        # 添加起点
        positions = [0] + sorted_locations
        costs = [0] + [supply_points[loc] for loc in sorted_locations]

        # dp[i] 表示在第 i 个补给站加满体力的最小花费
        dp = [float('inf')] * (num_valid_stations + 1)
        dp[0] = 0

        dq = deque()
        dq.append(0)

        for i in range(1, num_valid_stations + 1):
            # 移除无法到达当前站的队列前端
            while dq and positions[i] - positions[dq[0]] > stamina_max:
                dq.popleft()

            if dq:
                dp[i] = dp[dq[0]] + costs[i]

            # 保持队列单调递增
            while dq and dp[dq[-1]] >= dp[i]:
                dq.pop()
            dq.append(i)

        # 寻找最后一个可直接到达终点的站点
        min_total = float('inf')
        for i in range(num_valid_stations, -1, -1):
            if distance - positions[i] <= stamina_max:
                min_total = min(min_total, dp[i])
            else:
                break

        print("Yes" if min_total <= budget else "No")

if __name__ == "__main__":
    main()

## C 最长回文

In [ ]:
import java.io.*;

public class Main {
    static class PalindromeSolver {
        int length;
        String strA, strB;
        long[] pow1, pow2, hashB1, hashB2, revHashA1, revHashA2;
        final long MOD1 = 1_000_000_007L;
        final long MOD2 = 1_000_000_009L;
        final long BASE1 = 131L;
        final long BASE2 = 13331L;

        PalindromeSolver(int length, String strA, String strB) {
            this.length = length;
            this.strA = strA;
            this.strB = strB;
            pow1 = new long[length + 5];
            pow2 = new long[length + 5];
            hashB1 = new long[length + 5];
            hashB2 = new long[length + 5];
            revHashA1 = new long[length + 5];
            revHashA2 = new long[length + 5];

            pow1[0] = pow2[0] = 1;
            for (int i = 1; i <= length; i++) {
                pow1[i] = (pow1[i - 1] * BASE1) % MOD1;
                pow2[i] = (pow2[i - 1] * BASE2) % MOD2;
            }

            for (int i = 1; i <= length; i++) {
                hashB1[i] = (hashB1[i - 1] * BASE1 + strB.charAt(i - 1)) % MOD1;
                hashB2[i] = (hashB2[i - 1] * BASE2 + strB.charAt(i - 1)) % MOD2;
            }

            for (int i = 1; i <= length; i++) {
                char ch = strA.charAt(length - i);
                revHashA1[i] = (revHashA1[i - 1] * BASE1 + ch) % MOD1;
                revHashA2[i] = (revHashA2[i - 1] * BASE2 + ch) % MOD2;
            }
        }

        private boolean compareSubstrings(int aPos, int bPos, int len) {
            int aStart = length - aPos + 1;
            int aEnd = aStart + len - 1;
            int bStart = bPos;
            int bEnd = bPos + len - 1;

            long ha1 = (revHashA1[aEnd] - revHashA1[aStart - 1] * pow1[len]) % MOD1;
            if (ha1 < 0) ha1 += MOD1;
            long hb1 = (hashB1[bEnd] - hashB1[bStart - 1] * pow1[len]) % MOD1;
            if (hb1 < 0) hb1 += MOD1;
            if (ha1 != hb1) return false;

            long ha2 = (revHashA2[aEnd] - revHashA2[aStart - 1] * pow2[len]) % MOD2;
            if (ha2 < 0) ha2 += MOD2;
            long hb2 = (hashB2[bEnd] - hashB2[bStart - 1] * pow2[len]) % MOD2;
            if (hb2 < 0) hb2 += MOD2;

            return ha2 == hb2;
        }

        private int commonLength(int posA, int posB) {
            if (posA < 1 || posA > length || posB < 1 || posB > length) return 0;
            int left = 1, right = Math.min(posA, length - posB + 1), res = 0;
            while (left <= right) {
                int mid = (left + right) / 2;
                if (compareSubstrings(posA, posB, mid)) {
                    res = mid;
                    left = mid + 1;
                } else {
                    right = mid - 1;
                }
            }
            return res;
        }

        private int[] manacher(String s) {
            char[] t = new char[2 * length + 3];
            t[0] = '$'; t[1] = '#';
            for (int i = 0; i < length; i++) {
                t[2 * i + 2] = s.charAt(i);
                t[2 * i + 3] = '#';
            }
            t[2 * length + 2] = '^';

            int[] p = new int[t.length];
            int center = 0, right = 0;
            for (int i = 1; i < t.length - 1; i++) {
                int mirror = 2 * center - i;
                if (right > i) p[i] = Math.min(right - i, p[mirror]);
                else p[i] = 1;
                while (t[i + p[i]] == t[i - p[i]]) p[i]++;
                if (i + p[i] > right) {
                    center = i;
                    right = i + p[i];
                }
            }
            return p;
        }

        int findMaxPalindrome() {
            int[] P_A = manacher(strA);
            int[] P_B = manacher(strB);
            int maxLen = 0;

            for (int c = 1; c <= length; c++) {
                int radA = (P_A[2 * c] - 2) / 2;
                maxLen = Math.max(maxLen, 2 * radA + 1 + 2 * commonLength(c - radA - 1, c + radA));

                int radB = (P_B[2 * c] - 2) / 2;
                maxLen = Math.max(maxLen, 2 * radB + 1 + 2 * commonLength(c - radB, c + radB + 1));
            }

            for (int c = 0; c <= length; c++) {
                int radA = (P_A[2 * c + 1] - 1) / 2;
                maxLen = Math.max(maxLen, 2 * radA + 2 * commonLength(c - radA, c + radA));

                int radB = (P_B[2 * c + 1] - 1) / 2;
                maxLen = Math.max(maxLen, 2 * radB + 2 * commonLength(c - radB + 1, c + radB + 1));
            }

            return maxLen;
        }
    }

    public static void main(String[] args) throws IOException {
        BufferedReader reader = new BufferedReader(new InputStreamReader(System.in));
        String line;
        while ((line = reader.readLine()) != null) {
            line = line.trim();
            if (line.isEmpty()) continue;
            int n = Integer.parseInt(line);
            String aStr = reader.readLine().trim();
            String bStr = reader.readLine().trim();
            PalindromeSolver solver = new PalindromeSolver(n, aStr, bStr);
            System.out.println(solver.findMaxPalindrome());
        }
    }
}

## D 优惠券

In [ ]:
import java.io.*;
import java.util.*;

public class Main {

    static int[] currentState = new int[1000005];
    static int[] lastOccurrence = new int[1000005];
    static int[] visitedTest = new int[1000005];
    static int currentTestId = 0;
    static int[] segTree;

    static void initState(int id) {
        if (visitedTest[id] != currentTestId) {
            currentState[id] = 0;
            lastOccurrence[id] = 0;
            visitedTest[id] = currentTestId;
        }
    }

    static void updateSegment(int node, int left, int right, int pos, int val) {
        if (left == right) {
            segTree[node] = val;
            return;
        }
        int mid = (left + right) >> 1;
        if (pos <= mid) updateSegment(node << 1, left, mid, pos, val);
        else updateSegment((node << 1) | 1, mid + 1, right, pos, val);
        segTree[node] = segTree[node << 1] + segTree[(node << 1) | 1];
    }

    static int queryFirstSegment(int node, int left, int right, int ql, int qr) {
        if (ql > right || qr < left || segTree[node] == 0) return -1;
        if (left == right) return left;
        int mid = (left + right) >> 1;
        int res = -1;
        if (ql <= mid && segTree[node << 1] > 0) {
            res = queryFirstSegment(node << 1, left, mid, ql, qr);
        }
        if (res == -1 && qr > mid && segTree[(node << 1) | 1] > 0) {
            res = queryFirstSegment((node << 1) | 1, mid + 1, right, ql, qr);
        }
        return res;
    }

    static void clearSegment(int node, int left, int right) {
        if (segTree[node] == 0) return;
        segTree[node] = 0;
        if (left == right) return;
        int mid = (left + right) >> 1;
        clearSegment(node << 1, left, mid);
        clearSegment((node << 1) | 1, mid + 1, right);
    }

    public static void main(String[] args) {
        FastScanner scanner = new FastScanner();
        PrintWriter out = new PrintWriter(System.out);

        String token;
        while ((token = scanner.next()) != null) {
            int numOps = Integer.parseInt(token);
            currentTestId++;
            segTree = new int[4 * numOps + 5];

            int firstError = -1;
            boolean hasError = false;

            for (int i = 1; i <= numOps; i++) {
                String opToken = scanner.next();
                if (opToken == null) break;

                char firstChar = opToken.charAt(0);
                String operation;
                int val = 0;

                if (firstChar == 'I' || firstChar == 'O') {
                    operation = String.valueOf(firstChar);
                    if (opToken.length() > 1) val = Integer.parseInt(opToken.substring(1));
                    else val = Integer.parseInt(scanner.next());
                } else {
                    operation = "?";
                    // 清理可能的垃圾数字
                    while (true) {
                        String peeked = scanner.peek();
                        if (peeked != null) {
                            try { Integer.parseInt(peeked); scanner.next(); } 
                            catch (NumberFormatException e) { break; }
                        } else break;
                    }
                }

                if (hasError) continue;

                switch (operation) {
                    case "?":
                        updateSegment(1, 1, numOps, i, 1);
                        break;
                    case "I":
                        initState(val);
                        if (currentState[val] == 1) {
                            int q = -1;
                            if (lastOccurrence[val] + 1 <= i - 1) {
                                q = queryFirstSegment(1, 1, numOps, lastOccurrence[val] + 1, i - 1);
                            }
                            if (q != -1) {
                                updateSegment(1, 1, numOps, q, 0);
                                lastOccurrence[val] = i;
                            } else {
                                hasError = true;
                                firstError = i;
                            }
                        } else {
                            currentState[val] = 1;
                            lastOccurrence[val] = i;
                        }
                        break;
                    case "O":
                        initState(val);
                        if (currentState[val] == 0) {
                            int q = -1;
                            if (lastOccurrence[val] + 1 <= i - 1) {
                                q = queryFirstSegment(1, 1, numOps, lastOccurrence[val] + 1, i - 1);
                            }
                            if (q != -1) {
                                updateSegment(1, 1, numOps, q, 0);
                                lastOccurrence[val] = i;
                            } else {
                                hasError = true;
                                firstError = i;
                            }
                        } else {
                            currentState[val] = 0;
                            lastOccurrence[val] = i;
                        }
                        break;
                }
            }

            out.println(hasError ? firstError : -1);
            if (numOps > 0) clearSegment(1, 1, numOps);
        }
        out.flush();
    }

    static class FastScanner {
        BufferedReader br;
        StringTokenizer st;
        String nextToken = null;

        FastScanner() { br = new BufferedReader(new InputStreamReader(System.in)); }

        boolean hasNext() {
            if (nextToken != null) return true;
            while (st == null || !st.hasMoreElements()) {
                try {
                    String line = br.readLine();
                    if (line == null) return false;
                    st = new StringTokenizer(line);
                } catch (IOException e) { return false; }
            }
            nextToken = st.nextToken();
            return true;
        }

        String next() {
            if (!hasNext()) return null;
            String res = nextToken;
            nextToken = null;
            return res;
        }

        String peek() {
            if (!hasNext()) return null;
            return nextToken;
        }
    }
}

## E 任意点

In [ ]:
#include <bits/stdc++.h>
using namespace std;

struct Point {
    int x, y;
};

int find(vector<int>& parent, int u) {
    if (parent[u] != u) parent[u] = find(parent, parent[u]);
    return parent[u];
}

void unite(vector<int>& parent, int u, int v) {
    int pu = find(parent, u);
    int pv = find(parent, v);
    if (pu != pv) parent[pu] = pv;
}

// 判断两点是否可直接连通
bool connected(const Point &a, const Point &b) {
    return a.x == b.x || a.y == b.y;
}

int main() {
    int n;
    cin >> n;
    vector<Point> points(n);
    for (int i = 0; i < n; i++) {
        cin >> points[i].x >> points[i].y;
    }

    // 并查集初始化
    vector<int> parent(n);
    for (int i = 0; i < n; i++) parent[i] = i;

    // 合并连通的点
    for (int i = 0; i < n; i++) {
        for (int j = i+1; j < n; j++) {
            if (connected(points[i], points[j])) {
                unite(parent, i, j);
            }
        }
    }

    // 统计连通分量
    unordered_set<int> components;
    for (int i = 0; i < n; i++) {
        components.insert(find(parent, i));
    }
    
    cout << (int)components.size() - 1 << endl;
    return 0;
}

## F 通配符匹配

In [ ]:
#include <bits/stdc++.h>
using namespace std;

struct Segment {
    string str;
    vector<pair<int, string>> blocks;
};

Segment buildSegment(const string& t) {
    Segment seg;
    seg.str = t;

    int n = t.size();
    for (int i = 0; i < n; ) {
        if (t[i] == '?') {
            i++;
            continue;
        }

        int j = i;
        while (j < n && t[j] != '?') j++;

        seg.blocks.push_back({i, t.substr(i, j - i)});
        i = j;
    }

    return seg;
}

bool matchAt(const string& s, int pos, const Segment& seg) {
    int n = s.size();
    int len = seg.str.size();

    if (pos < 0 || pos + len > n) return false;

    for (auto &b : seg.blocks) {
        int offset = b.first;
        const string& part = b.second;

        if (s.compare(pos + offset, part.size(), part) != 0) {
            return false;
        }
    }

    return true;
}

int findSegment(const string& s, const Segment& seg, int start, int maxStart) {
    int len = seg.str.size();

    if (start > maxStart) return -1;

    // 如果这一段全是 '?'
    if (seg.blocks.empty()) {
        if (start + len <= (int)s.size()) return start;
        return -1;
    }

    // 选择最长的普通字符串块来加速查找
    int bestId = 0;
    for (int i = 1; i < (int)seg.blocks.size(); i++) {
        if (seg.blocks[i].second.size() > seg.blocks[bestId].second.size()) {
            bestId = i;
        }
    }

    int offset = seg.blocks[bestId].first;
    const string& key = seg.blocks[bestId].second;

    int searchPos = max(0, start + offset);

    while (true) {
        size_t found = s.find(key, searchPos);
        if (found == string::npos) return -1;

        int pos = (int)found - offset;

        if (pos > maxStart) return -1;

        if (pos >= start && matchAt(s, pos, seg)) {
            return pos;
        }

        searchPos = found + 1;
    }
}

bool check(const string& pattern, const string& filename) {
    vector<string> raw;
    string cur;

    bool hasStar = false;

    for (char c : pattern) {
        if (c == '*') {
            hasStar = true;
            raw.push_back(cur);
            cur.clear();
        } else {
            cur.push_back(c);
        }
    }
    raw.push_back(cur);

    // 没有 '*': 长度必须完全相等
    if (!hasStar) {
        if (pattern.size() != filename.size()) return false;
        Segment seg = buildSegment(pattern);
        return matchAt(filename, 0, seg);
    }

    vector<Segment> segs;
    for (auto &x : raw) {
        segs.push_back(buildSegment(x));
    }

    int n = filename.size();

    int current = 0;

    // 处理开头固定部分
    int left = 0;
    if (!pattern.empty() && pattern[0] != '*') {
        if (!matchAt(filename, 0, segs[0])) return false;
        current = segs[0].str.size();
        left = 1;
    }

    // 处理结尾固定部分
    int right = (int)segs.size() - 1;
    int suffixStart = n;

    if (!pattern.empty() && pattern.back() != '*') {
        int len = segs.back().str.size();

        if (len > n) return false;

        suffixStart = n - len;

        if (!matchAt(filename, suffixStart, segs.back())) {
            return false;
        }

        right--;
    }

    // 中间部分按顺序匹配即可
    for (int i = left; i <= right; i++) {
        if (segs[i].str.empty()) continue;

        int len = segs[i].str.size();
        int maxStart = suffixStart - len;

        int pos = findSegment(filename, segs[i], current, maxStart);

        if (pos == -1) return false;

        current = pos + len;
    }

    return current <= suffixStart;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string pattern;
    cin >> pattern;

    int n;
    cin >> n;

    while (n--) {
        string filename;
        cin >> filename;

        if (check(pattern, filename)) {
            cout << "YES\n";
        } else {
            cout << "NO\n";
        }
    }

    return 0;
}

## G 汉诺塔

In [ ]:
#include <bits/stdc++.h>
using namespace std;

long long simulate3(const vector<string>& ops) {
    int n = 3;

    vector<int> pos(n, 0); 
    // pos[i] 表示第 i 个盘子在哪根柱子上
    // 0 表示 A，1 表示 B，2 表示 C
    // i 越小，盘子越小

    int last = -1;
    long long step = 0;

    map<string, int> priority;
    for (int i = 0; i < 6; i++) {
        priority[ops[i]] = i;
    }

    while (true) {
        bool allB = true, allC = true;
        for (int i = 0; i < n; i++) {
            if (pos[i] != 1) allB = false;
            if (pos[i] != 2) allC = false;
        }

        if (allB || allC) return step;

        vector<int> top(3, -1);

        for (int i = 0; i < n; i++) {
            int p = pos[i];
            if (top[p] == -1) {
                top[p] = i;
            }
        }

        string best = "";
        int bestPriority = 10;
        int moveDisk = -1;

        for (string op : ops) {
            int from = op[0] - 'A';
            int to = op[1] - 'A';

            int disk = top[from];

            if (disk == -1) continue;
            if (disk == last) continue;

            if (top[to] == -1 || disk < top[to]) {
                if (priority[op] < bestPriority) {
                    bestPriority = priority[op];
                    best = op;
                    moveDisk = disk;
                }
            }
        }

        int to = best[1] - 'A';
        pos[moveDisk] = to;
        last = moveDisk;
        step++;
    }
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;

    vector<string> ops(6);
    for (int i = 0; i < 6; i++) {
        cin >> ops[i];
    }

    long long cnt3 = simulate3(ops);

    long long ans = 0;

    if (cnt3 == 7) {
        ans = (1LL << n) - 1;
    } else {
        long long p = 1;
        for (int i = 1; i <= n - 1; i++) {
            p *= 3;
        }

        if (cnt3 == 9) {
            ans = p;
        } else {
            ans = 2 * p - 1;
        }
    }

    cout << ans << '\n';

    return 0;
}

## H 马步距离

In [ ]:
#include <bits/stdc++.h>
using namespace std;

int main() {
    long long xp, yp, xs, ys;
    cin >> xp >> yp >> xs >> ys;

    long long dx = llabs(xs - xp);
    long long dy = llabs(ys - yp);

    long long x = max(dx, dy);
    long long y = min(dx, dy);

    if (x == 0 && y == 0) {
        cout << 0 << endl;
        return 0;
    }

    if (x == 1 && y == 0) {
        cout << 3 << endl;
        return 0;
    }

    if (x == 2 && y == 2) {
        cout << 4 << endl;
        return 0;
    }

    long long ans = max((x + 1) / 2, (x + y + 2) / 3);

    if ((ans + x + y) % 2 != 0) {
        ans++;
    }

    cout << ans << endl;

    return 0;
}

## I 直方图最大矩形

In [ ]:
class Solution {
public:
    int largestRectangleArea(vector<int>& heights) {
        // write code here
        vector<int> h;
        h.push_back(0);

        for (int x : heights) {
            h.push_back(x);
        }

        h.push_back(0);

        stack<int> st;
        int ans = 0;

        for (int i = 0; i < h.size(); i++) {
            while (!st.empty() && h[i] < h[st.top()]) {
                int height = h[st.top()];
                st.pop();

                int left = st.top();
                int right = i;

                int width = right - left - 1;
                ans = max(ans, height * width);
            }

            st.push(i);
        }

        return ans;
    }
};

## J 消防局的设立

In [ ]:
#include <bits/stdc++.h>
using namespace std;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;

    vector<vector<int>> g(n + 1);

    for (int i = 2; i <= n; i++) {
        int p;
        cin >> p;
        g[i].push_back(p);
        g[p].push_back(i);
    }

    vector<int> parent(n + 1, 0), depth(n + 1, 0), order;
    order.reserve(n);

    queue<int> q;
    q.push(1);
    parent[1] = 0;

    while (!q.empty()) {
        int u = q.front();
        q.pop();
        order.push_back(u);

        for (int v : g[u]) {
            if (v == parent[u]) continue;
            parent[v] = u;
            depth[v] = depth[u] + 1;
            q.push(v);
        }
    }

    vector<bool> station(n + 1, false);
    vector<bool> childStation(n + 1, false);
    vector<bool> grandChildStation(n + 1, false);

    auto covered = [&](int u) {
        int p = parent[u];
        int gp = p ? parent[p] : 0;

        if (station[u]) return true;
        if (p && station[p]) return true;
        if (gp && station[gp]) return true;

        if (childStation[u]) return true;
        if (grandChildStation[u]) return true;

        // 兄弟节点有消防局，距离也是 2
        if (p && childStation[p]) return true;

        return false;
    };

    int ans = 0;

    for (int i = n - 1; i >= 0; i--) {
        int u = order[i];

        if (covered(u)) continue;

        int pos = u;

        if (parent[pos]) pos = parent[pos];
        if (parent[pos]) pos = parent[pos];

        station[pos] = true;
        ans++;

        int p = parent[pos];
        if (p) {
            childStation[p] = true;

            int gp = parent[p];
            if (gp) {
                grandChildStation[gp] = true;
            }
        }
    }

    cout << ans << '\n';

    return 0;
}